# Week 2 — Class 4: Numerical Processing with NumPy
## Interactive Lecture Notebook

**Run each cell sequentially** to see concepts in action.

| # | Topic |
| --- | --- |
| 1 | N-Dimensional Arrays (`ndarray`) |
| 2 | Vectorization vs. Loops |
| 3 | Dot Products & Matrix Multiplication |
| 4 | Capstone: Build a Similarity Engine |

---

# 1. N-Dimensional Arrays (ndarray)

### Topic
NumPy-এর কোর ডেটা স্ট্রাকচার `ndarray` — নিউমেরিক্যাল ডেটার **Universal Container**।

### Why It Is Related
এআই সিস্টেমের পেছনে সবকিছুই সংখ্যা:
- **Image** = 3D array `(height × width × channels)`
- **Text embeddings** = 2D array `(tokens × dimensions)`
- **Model weights** = multi-dimensional tensors

PyTorch / TensorFlow / JAX — সবার ভিত্তি এই NumPy। শেপ ডিবাগ করা, মেমোরি ম্যানেজ করা, এআই কোড রিড করা — সব এখান থেকে শুরু।

### How It Works
`ndarray` = মেমোরির একটি **contiguous block**, যাতে **homogeneous** ডেটা (সব `float64` বা সব `int32`) থাকে। পাইথন লিস্টের মতো ছড়ানো অবজেক্টের পয়েন্টার নয় — raw value গুলো **C-order layout**-এ পাশাপাশি বসানো। তাই compiled C-speed-এ vectorized অপারেশন সম্ভব।

**৪টি কোর প্রপার্টি:**
- **Shape** — ডাইমেনশনের tuple, যেমন `(batch_size, seq_len, embed_dim)`
- **Dtype** — প্রতি এলিমেন্ট কত বাইট নেবে (`float32`, `int64`)
- **Strides** — এক axis থেকে পরের axis-এ যেতে কত বাইট jump
- **Broadcasting** — ছোট শেপের array কে বড় শেপে auto-expand

---

### Analogy
🗄️ **Python List = আলাদা আলাদা ফোল্ডারওয়ালা ফাইল কেবিনেট** — প্রতি ফোল্ডার মেমোরির আলাদা জায়গায়। যোগ করতে হলে প্রতিটা খুলে, কাগজ বের করে, আবার রাখতে হবে। Slow, expensive।

📗 **NumPy Array = Excel Spreadsheet** — সব ডেটা এক গ্রিডে পাশাপাশি। দুই শিট যোগ করা মানে একটাকে আরেকটার ওপর বসিয়ে এক ক্লিক। Fast, organized, vectorized।

---

## 1.1 Creating Arrays & Understanding Structure

**RUN THE CELL BELOW** 👇

In [ ]:
import numpy as np
import sys

# 1D — a single embedding vector
vec = np.array([1.0, 2.0, 3.0, 4.0])

# 2D — a batch of embeddings (tokens x dimensions)
mat = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])
print()


# 3D — an RGB image (height x width x channels)
img = np.zeros((4, 4, 3), dtype=np.uint8)
# print(img)

for name, arr in [('vec (1D)', vec), ('mat (2D)', mat), ('img (3D)', img)]:
    print(f"{name}")
    print(f"  shape   : {arr.shape}")
    print(f"  ndim    : {arr.ndim}")
    print(f"  dtype   : {arr.dtype}")
    print(f"  itemsize: {arr.itemsize} bytes/element")
    print(f"  strides : {arr.strides}  <- bytes to jump per axis")
    print(f"  nbytes  : {arr.nbytes} bytes")
    print()

# STRIDES explained on mat (2 rows x 3 cols, float64 = 8 bytes)
# strides = (24, 8):  next ROW is 24 bytes away (3 cols x 8), next COL is 8 bytes away.
# That is C-order (row-major): the LAST axis is contiguous in memory.
print(f"mat is C-contiguous? {mat.flags['C_CONTIGUOUS']}")
print(f"mat.T strides     : {mat.T.strides}  <- transpose just swaps strides, copies NOTHING")
print(f"mat.T C-contiguous? {mat.T.flags['C_CONTIGUOUS']}")

# ANALOGY: strides = 'how many steps to the next shelf' in the warehouse.
#          Transpose = re-labelling the aisles, not moving the boxes.

vec (1D)
  shape   : (4,)
  ndim    : 1
  dtype   : float64
  itemsize: 8 bytes/element
  strides : (8,)  <- bytes to jump per axis
  nbytes  : 32 bytes

mat (2D)
  shape   : (2, 3)
  ndim    : 2
  dtype   : float64
  itemsize: 8 bytes/element
  strides : (24, 8)  <- bytes to jump per axis
  nbytes  : 48 bytes

img (3D)
  shape   : (4, 4, 3)
  ndim    : 3
  dtype   : uint8
  itemsize: 1 bytes/element
  strides : (12, 3, 1)  <- bytes to jump per axis
  nbytes  : 48 bytes

mat is C-contiguous? True
mat.T strides     : (8, 24)  <- transpose just swaps strides, copies NOTHING
mat.T C-contiguous? False


## 1.2 Memory Efficiency — Why 28MB vs 8MB Decides GPU or OOM

**RUN THE CELL BELOW** 👇

In [8]:
n = 1_000_000

py_list = [float(i) for i in range(n)]

np_arr = np.arange(n, dtype=np.float64)



# Python list: pointer array + a separate float OBJECT per element
list_bytes = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list[:1000]) / 1000 * n

print(f"1M floats")
print(f"  Python list : {list_bytes / 1024**2:6.1f} MB  (pointers + per-object overhead)")
print(f"  NumPy float64: {np_arr.nbytes / 1024**2:6.1f} MB  (raw values, back to back)")
print(f"  NumPy float32: {np_arr.astype(np.float32).nbytes / 1024**2:6.1f} MB  (half precision)")

# WHY this matters: GPT-4 class models carry ~1.8T parameters.
# The dtype choice alone is the difference between fitting on the GPU
# and crashing with OOM (Out of Memory).
print(f"\n1.8T params, memory needed just for weights:")
for dt, size in [('float64', 8), ('float32', 4), ('float16', 2), ('int8', 1)]:
    print(f"  {dt:8s}: {1.8e12 * size / 1024**4:7.1f} TB")

# ANALOGY: list = every number wrapped in its own gift box on a shelf.
#          array = the numbers poured straight into a measuring tube.

1M floats
  Python list :   30.9 MB  (pointers + per-object overhead)
  NumPy float64:    7.6 MB  (raw values, back to back)
  NumPy float32:    3.8 MB  (half precision)

1.8T params, memory needed just for weights:
  float64 :    13.1 TB
  float32 :     6.5 TB
  float16 :     3.3 TB
  int8    :     1.6 TB


## 1.3 Views vs. Copies — The Silent Bug Factory

**RUN THE CELL BELOW** 👇

In [9]:
original = np.arange(10)
print(f"original      : {original}")

# SLICING returns a VIEW — shares the same memory buffer
view = original[0:5]
view[0] = 999
print(f"after view[0]=999")
print(f"  view        : {view}")
print(f"  original    : {original}   <- ORIGINAL CHANGED TOO!")
print(f"  shares memory: {np.shares_memory(view, original)}")
print(f"  view.base is original: {view.base is original}")

# .copy() breaks the link
original = np.arange(10)
copy = original[0:5].copy()
copy[0] = 999
print(f"\nafter copy[0]=999")
print(f"  copy        : {copy}")
print(f"  original    : {original}   <- untouched")
print(f"  shares memory: {np.shares_memory(copy, original)}")

# Fancy indexing ALWAYS copies, basic slicing NEVER does
arr = np.arange(10)
print(f"\nbasic slice arr[2:5]      -> view? {np.shares_memory(arr[2:5], arr)}")
print(f"fancy index arr[[2,3,4]]  -> view? {np.shares_memory(arr[[2, 3, 4]], arr)}")
print(f"bool mask   arr[arr > 5]  -> view? {np.shares_memory(arr[arr > 5], arr)}")

# WHEN to use which?
print("\nGUIDELINES:")
print("  VIEW  -> batching a huge dataset, avoid OOM (zero extra memory)")
print("  COPY  -> before in-place modification, avoid corrupting the source")

# ANALOGY: view = a window cut into the wall of the warehouse.
#          copy = renting a second warehouse and hauling the boxes over.

original      : [0 1 2 3 4 5 6 7 8 9]
after view[0]=999
  view        : [999   1   2   3   4]
  original    : [999   1   2   3   4   5   6   7   8   9]   <- ORIGINAL CHANGED TOO!
  shares memory: True
  view.base is original: True

after copy[0]=999
  copy        : [999   1   2   3   4]
  original    : [0 1 2 3 4 5 6 7 8 9]   <- untouched
  shares memory: False

basic slice arr[2:5]      -> view? True
fancy index arr[[2,3,4]]  -> view? False
bool mask   arr[arr > 5]  -> view? False

GUIDELINES:
  VIEW  -> batching a huge dataset, avoid OOM (zero extra memory)
  COPY  -> before in-place modification, avoid corrupting the source


## 1.4 Broadcasting — Adding a Bias Vector to a Whole Batch

**Rule:** দুটি শেপ ডানদিক থেকে align করা হয়। প্রতি axis-এ হয় দুটো সমান, নাহয় একটি `1`।

**RUN THE CELL BELOW** 👇

In [8]:
# A batch of 4 samples, each a 3-dim embedding
batch = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [10., 11., 12.],
])
bias = np.array([100.0, 200.0, 300.0])   # shape (3,)

print(f"batch shape: {batch.shape}")
print(f"bias  shape: {bias.shape}")
print(f"batch + bias -> shape {(batch + bias).shape}\n{batch + bias}")
print("\nHOW: (4,3) and (3,) align right -> (4,3) vs (1,3) -> bias repeated 4x VIRTUALLY.")
print("     No 4x memory is allocated. NumPy just uses stride 0 on that axis.")

# Per-sample scaling needs an explicit column shape
scale = np.array([1.0, 10.0, 100.0, 1000.0])         # shape (4,)
print(f"\nbatch * scale[:, None] -> per-ROW scaling, shape (4,1) vs (4,3)")
print(batch * scale[:, None])

# Broadcasting failure — the classic shape bug
try:
    batch + np.array([1.0, 2.0, 3.0, 4.0])           # (4,3) vs (4,) -> right-align fails
except ValueError as e:
    print(f"\nShape error: {e}")
    print("  FIX: reshape to (4,1) with [:, None] so the axes line up.")

# Outer product via broadcasting: (3,1) x (1,4) -> (3,4)
a = np.array([1, 2, 3])
b = np.array([10, 20, 30, 40])
print(f"\na[:,None] * b[None,:] -> {(a[:, None] * b[None, :]).shape}")
print(a[:, None] * b[None, :])

# ANALOGY: broadcasting = a rubber stamp.
#          One stamp (bias), pressed onto every page (row), no photocopies made.

batch shape: (4, 3)
bias  shape: (3,)
batch + bias -> shape (4, 3)
[[101. 202. 303.]
 [104. 205. 306.]
 [107. 208. 309.]
 [110. 211. 312.]]

HOW: (4,3) and (3,) align right -> (4,3) vs (1,3) -> bias repeated 4x VIRTUALLY.
     No 4x memory is allocated. NumPy just uses stride 0 on that axis.

batch * scale[:, None] -> per-ROW scaling, shape (4,1) vs (4,3)
[[1.0e+00 2.0e+00 3.0e+00]
 [4.0e+01 5.0e+01 6.0e+01]
 [7.0e+02 8.0e+02 9.0e+02]
 [1.0e+04 1.1e+04 1.2e+04]]

Shape error: operands could not be broadcast together with shapes (4,3) (4,) 
  FIX: reshape to (4,1) with [:, None] so the axes line up.

a[:,None] * b[None,:] -> (3, 4)
[[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]


---

# 2. Vectorization vs. Loops

### Topic
পাইথনের ধীরগতির `for` লুপ বাদ দিয়ে NumPy-এর **vectorized operations** দিয়ে massive speed gain।

### Why It Is Related
একটি Transformer-এর single forward pass-এ প্রায় $10^{12}$ multiply-add অপারেশন লাগে। পাইথন লুপে করলে একটা মডেল ট্রেইন করতে শতাব্দী লেগে যেত। Vectorization পুরো কম্পিউটেশন অপ্টিমাইজড **C / BLAS** লাইব্রেরিতে পাঠায়।

### How It Works
- **Python loop** — element-by-element; প্রতি iteration-এ interpreter overhead + type check
- **NumPy vectorized** — এক লাইনে compiled C কল; প্রসেসরের **SIMD** / AVX-512 ইন্সট্রাকশন ব্যবহার করে একসাথে অনেক ডেটা
- **ufuncs** — `np.add`, `np.multiply`, `np.sqrt` পুরো array একবারে হ্যান্ডেল করে
- **Axis parameter** — `axis=0` কলাম বরাবর, `axis=1` রো বরাবর। ব্যাচ অপারেশনের প্রাণ

### Benchmark (1M elements)
| Approach | Time |
| --- | --- |
| Python loop | ~200 ms |
| NumPy vectorized | ~1 ms |
| GPU (PyTorch) | ~0.1 ms |

লুপ → NumPy গ্যাপ **200x**, কিন্তু NumPy → GPU গ্যাপ মাত্র **10x**। তাই কোড ফাস্ট করার সবচেয়ে বড় হাতিয়ার হলো vectorization, GPU নয়।

---

### Analogy
🪣 **১০০০টি বেড়ার কাঠ রঙ করা:**
- **Python loop** — ছোট ব্রাশ, এক কাঠ রঙ করে বালতিতে ফেরত, আবার পরেরটা। ১০০০ বার আসা-যাওয়া।
- **NumPy** — **spray gun**। এক টানে অনেক কাঠ। ট্যাংক = C library, আপনার হাত (Python) শুধু direction দেয়।
- **GPU** — **হেলিকপ্টার** থেকে পুরো দেয়াল একসাথে। সুপার ফাস্ট, কিন্তু আলাদা fuel (GPU memory) + license (CUDA) লাগে।

---

## 2.1 Speed Benchmark — Measure It Yourself

**RUN THE CELL BELOW** 👇

In [9]:
import time

n = 1_000_000
a_list = list(range(n))
b_list = list(range(n))
a = np.arange(n, dtype=np.float64)
b = np.arange(n, dtype=np.float64)

def timeit(label, fn):
    t0 = time.perf_counter()
    out = fn()
    ms = (time.perf_counter() - t0) * 1000
    print(f"  {label:28s}: {ms:8.2f} ms")
    return ms, out

print(f"Element-wise add of {n:,} numbers:")
t_loop, _ = timeit("pure python for-loop", lambda: [a_list[i] + b_list[i] for i in range(n)])
t_zip, _ = timeit("list comp + zip", lambda: [x + y for x, y in zip(a_list, b_list)])
t_np, _ = timeit("numpy vectorized a + b", lambda: a + b)

print(f"\n  NumPy is {t_loop / t_np:.0f}x faster than the for-loop")

# A heavier, more realistic op: sqrt(a^2 + b^2)
print(f"\nsqrt(a^2 + b^2):")
import math
t_loop2, _ = timeit("python loop + math.sqrt", lambda: [math.sqrt(x * x + y * y) for x, y in zip(a_list, b_list)])
t_np2, _ = timeit("np.sqrt(a**2 + b**2)", lambda: np.sqrt(a**2 + b**2))
print(f"\n  Speedup: {t_loop2 / t_np2:.0f}x")

# WHY the gap? Per element the loop pays for:
#   bytecode dispatch + type check + PyObject unbox + re-box + refcount.
# NumPy pays that ONCE for the whole array, then runs a tight C loop with SIMD.

Element-wise add of 1,000,000 numbers:
  pure python for-loop        :    37.22 ms
  list comp + zip             :    20.79 ms
  numpy vectorized a + b      :     1.94 ms

  NumPy is 19x faster than the for-loop

sqrt(a^2 + b^2):
  python loop + math.sqrt     :    62.98 ms
  np.sqrt(a**2 + b**2)        :     1.85 ms

  Speedup: 34x


## 2.2 ufuncs & the `axis` Parameter

**RUN THE CELL BELOW** 👇

In [10]:
np.random.seed(42)
# A batch of 4 samples, 5 features each — the shape you meet everywhere in AI
X = np.random.randint(1, 10, size=(4, 5)).astype(np.float64)
print(f"X (batch=4, features=5):\n{X}\n")

# AXIS: which axis gets COLLAPSED
print(f"X.sum()          = {X.sum():.0f}          <- everything -> scalar")
print(f"X.sum(axis=0)    = {X.sum(axis=0)}  <- collapse ROWS   -> per-FEATURE total, shape {X.sum(axis=0).shape}")
print(f"X.sum(axis=1)    = {X.sum(axis=1)}  <- collapse COLS   -> per-SAMPLE total, shape {X.sum(axis=1).shape}")
print(f"X.sum(axis=1, keepdims=True).shape = {X.sum(axis=1, keepdims=True).shape}  <- keeps it broadcastable\n")

# Real AI use: feature standardisation (z-score per feature, i.e. down the batch axis)
mu = X.mean(axis=0, keepdims=True)    # (1,5)
sd = X.std(axis=0, keepdims=True)     # (1,5)
X_std = (X - mu) / sd                 # broadcasting does the work
print(f"Standardised per feature:\n{X_std.round(2)}")
print(f"  new mean per feature: {X_std.mean(axis=0).round(6)}")
print(f"  new std  per feature: {X_std.std(axis=0).round(6)}\n")

# ufuncs — one call, whole array, C speed
print(f"np.sqrt(X)[0]  = {np.sqrt(X)[0].round(2)}")
print(f"np.exp(X_std)[0] = {np.exp(X_std)[0].round(3)}")
print(f"np.maximum(X, 5)[0] = {np.maximum(X, 5)[0]}   <- this IS ReLU when the floor is 0\n")

# Softmax over the last axis — the exact pattern used in attention
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)   # subtract max for numerical stability
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

P = softmax(X)
print(f"softmax(X) row sums = {P.sum(axis=1)}   <- each row is a probability distribution")

# Boolean masking is vectorized too — no loop, no if-statement
print(f"\nX > 5 mask:\n{X > 5}")
print(f"count of X > 5 : {(X > 5).sum()}")
print(f"np.where(X>5, X, 0)[0] = {np.where(X > 5, X, 0)[0]}")

# ANALOGY: axis = which direction you squash the sponge.
#          axis=0 squashes top-to-bottom (one value per column).

X (batch=4, features=5):
[[7. 4. 8. 5. 7.]
 [3. 7. 8. 5. 4.]
 [8. 8. 3. 6. 5.]
 [2. 8. 6. 2. 5.]]

X.sum()          = 111          <- everything -> scalar
X.sum(axis=0)    = [20. 27. 25. 18. 21.]  <- collapse ROWS   -> per-FEATURE total, shape (5,)
X.sum(axis=1)    = [31. 27. 30. 23.]  <- collapse COLS   -> per-SAMPLE total, shape (4,)
X.sum(axis=1, keepdims=True).shape = (4, 1)  <- keeps it broadcastable

Standardised per feature:
[[ 0.78 -1.68  0.86  0.33  1.61]
 [-0.78  0.15  0.86  0.33 -1.15]
 [ 1.18  0.76 -1.59  1.   -0.23]
 [-1.18  0.76 -0.12 -1.67 -0.23]]
  new mean per feature: [ 0.  0.  0. -0. -0.]
  new std  per feature: [1. 1. 1. 1. 1.]

np.sqrt(X)[0]  = [2.65 2.   2.83 2.24 2.65]
np.exp(X_std)[0] = [2.191 0.187 2.352 1.396 4.982]
np.maximum(X, 5)[0] = [7. 5. 8. 5. 7.]   <- this IS ReLU when the floor is 0

softmax(X) row sums = [1. 1. 1. 1.]   <- each row is a probability distribution

X > 5 mask:
[[ True False  True False  True]
 [False  True  True False False]
 [ True  Tr

## 2.3 Memory Layout — Why `axis=-1` Is the Fast One

NumPy ডিফল্টে **row-major (C-order)**। শেষ axis বরাবর ডেটা মেমোরিতে গায়ে গায়ে লাগানো, তাই সেই দিকের অপারেশন cache-friendly ও দ্রুততম।

**RUN THE CELL BELOW** 👇

In [11]:
big = np.random.rand(4000, 4000)          # C-order (row-major)
big_f = np.asfortranarray(big)             # same numbers, F-order (column-major)

def ms(fn, repeat=3):
    fn()                                    # warm the cache
    t0 = time.perf_counter()
    for _ in range(repeat):
        fn()
    return (time.perf_counter() - t0) * 1000 / repeat

# The FAIR test: identical reduction, identical numbers, only the LAYOUT differs.
print("sum(axis=1) - scans along the last axis")
print(f"  C-order (last axis contiguous)  : {ms(lambda: big.sum(axis=1)):7.2f} ms")
print(f"  F-order (last axis strided)     : {ms(lambda: big_f.sum(axis=1)):7.2f} ms   <- cache misses")
print("\nsum(axis=0) - scans down the first axis")
print(f"  C-order (first axis strided)    : {ms(lambda: big.sum(axis=0)):7.2f} ms")
print(f"  F-order (first axis contiguous) : {ms(lambda: big_f.sum(axis=0)):7.2f} ms")

print(f"\nstrides -> C: {big.strides},  F: {big_f.strides}")
print("RULE: a reduction is fastest when the data it walks over is contiguous.")
print("CAVEAT: on a C-order array sum(axis=0) can still beat sum(axis=1) in raw time,")
print("        because NumPy adds whole contiguous ROWS into an accumulator with SIMD,")
print("        while axis=1 reduces inside each row. Layout is what matters, not the axis number.")

# A transposed view is F-contiguous - same data, reversed strides
bigT = big.T
print(f"\nbig.T -> strides {bigT.strides}, C-contig {bigT.flags['C_CONTIGUOUS']}, F-contig {bigT.flags['F_CONTIGUOUS']}")
print(f"  np.shares_memory(big, big.T) = {np.shares_memory(big, bigT)}  <- transpose copies nothing")

# np.ascontiguousarray() forces a real copy into C-order - pay once, read fast many times
t0 = time.perf_counter(); bigT_c = np.ascontiguousarray(bigT); t_copy = (time.perf_counter() - t0) * 1000
print(f"  ascontiguousarray copy cost: {t_copy:.2f} ms (worth it if you scan it repeatedly)")

# WHY it matters: a Transformer computes attention across the FEATURE dimension,
# which is the last axis of a C-order tensor. That is not a coincidence.

# ANALOGY: reading contiguous memory = reading a book line by line.
#          reading strided memory = reading the first word of every page, then the second.

sum(axis=1) - scans along the last axis
  C-order (last axis contiguous)  :    2.99 ms
  F-order (last axis strided)     :    2.45 ms   <- cache misses

sum(axis=0) - scans down the first axis
  C-order (first axis strided)    :    4.08 ms
  F-order (first axis contiguous) :    2.72 ms

strides -> C: (32000, 8),  F: (8, 32000)
RULE: a reduction is fastest when the data it walks over is contiguous.
CAVEAT: on a C-order array sum(axis=0) can still beat sum(axis=1) in raw time,
        because NumPy adds whole contiguous ROWS into an accumulator with SIMD,
        while axis=1 reduces inside each row. Layout is what matters, not the axis number.

big.T -> strides (8, 32000), C-contig False, F-contig True
  np.shares_memory(big, big.T) = True  <- transpose copies nothing
  ascontiguousarray copy cost: 54.01 ms (worth it if you scan it repeatedly)


---

# 3. Dot Products and Matrix Multiplication

### Topic
লিনিয়ার অ্যালজেব্রার দুই কোর অপারেশন যা প্রতিটি নিউরাল নেটওয়ার্ক চালায়: **dot product**, **matrix multiplication**, এবং এদের geometric অর্থ।

### Why It Is Related
নিউরাল নেটওয়ার্ক = matrix multiplication + non-linear activation-এর cascade। LLM-এর কোর টেকনোলজি **Self-Attention** আসলে একটি *scaled dot-product similarity*। Embedding-এর মিল মাপা হয় dot product দিয়ে। এটা না বুঝলে এআই কীভাবে "চিন্তা করে" বোঝা অসম্ভব।

### How It Works
- **Dot product:** $a \cdot b = \sum_i a_i b_i$ — দুই ভেক্টরের alignment মাপে। একই দিক → বড় positive; orthogonal → `0`; বিপরীত → negative।
- **Matmul:** $(A@B)_{ij} = \sum_k A_{ik} B_{kj}$, রুল $(m,n) @ (n,p) = (m,p)$। ভেতরের ডাইমেনশন অবশ্যই ম্যাচ করবে। পাইথনে `@` অপারেটর।
- **Geometric meaning:** $a \cdot b = |a||b|\cos\theta$। নরমালাইজ করলে **cosine similarity** (−1 থেকে 1)। Semantic search-এর ভিত্তি।
- **Batched matmul:** $(b,m,n) @ (b,n,p) = (b,m,p)$ — Transformer-এর প্রতি লেয়ারে অনবরত চলে।

### Deep Dive
- **Attention as matmul:** `Q @ K.T` দেয় `(seq_len, seq_len)` ম্যাট্রিক্স — token-to-token similarity। মানে: "token `i` token `j`-কে কতটুকু গুরুত্ব দেবে?" এরপর softmax, তারপর `@ V` — final output।
- **Complexity:** naive matmul $O(n^3)$, কিন্তু cuBLAS / Intel MKL tile-based blocking + parallelism দিয়ে বাস্তবে অনেক ফাস্ট।

---

### Analogy
🗳️ **Dot product = Voting system.** দুই ভেক্টর = দুই ভোটার। প্রতি ডাইমেনশন = একটি ইস্যু (অর্থনীতি, স্বাস্থ্য, জলবায়ু)। Dot product গোনে তারা কত ইস্যুতে একমত এবং কত তীব্রতায়। একই পলিসি strongly support → high score = similar। সব বিষয়ে দ্বিমত → negative।

🏭 **Matmul = Supply chain.** A = ফ্যাক্টরি × কাঁচামাল। B = কাঁচামাল × প্রোডাক্ট। `A @ B` = ফ্যাক্টরি × প্রোডাক্ট। ভেতরের ডাইমেনশন (কাঁচামাল) হলো পাইপলাইন — দুই পাশে সমান হতেই হবে।

---

## 3.1 Dot Product & Its Geometry

**RUN THE CELL BELOW** 👇

In [12]:
a = np.array([3.0, 4.0])
b = np.array([4.0, 3.0])

print(f"a = {a}, b = {b}")
print(f"manual sum(a_i * b_i) = {sum(x * y for x, y in zip(a, b))}")
print(f"np.dot(a, b)          = {np.dot(a, b)}")
print(f"a @ b                 = {a @ b}")
print(f"(a * b).sum()         = {(a * b).sum()}   <- elementwise multiply THEN reduce\n")

# Geometry: a . b = |a||b| cos(theta)
def angle_deg(u, v):
    cos = (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))
    return np.degrees(np.arccos(np.clip(cos, -1, 1))), cos

cases = {
    'same direction  ': (np.array([1.0, 0.0]), np.array([5.0, 0.0])),
    'orthogonal      ': (np.array([1.0, 0.0]), np.array([0.0, 5.0])),
    'opposite        ': (np.array([1.0, 0.0]), np.array([-5.0, 0.0])),
    '45 degrees      ': (np.array([1.0, 0.0]), np.array([1.0, 1.0])),
}
print(f"{'case':18s} {'dot':>8s} {'cos':>8s} {'angle':>8s}")
for label, (u, v) in cases.items():
    deg, cos = angle_deg(u, v)
    print(f"{label} {u @ v:8.2f} {cos:8.2f} {deg:7.1f}°")

# KEY INSIGHT: raw dot product is polluted by MAGNITUDE.
# Cosine similarity divides it out, so only DIRECTION (meaning) remains.
short = np.array([1.0, 1.0])
long_ = np.array([100.0, 100.0])
other = np.array([1.0, 1.0])
print(f"\nsame direction, different magnitude:")
print(f"  dot(short, other) = {short @ other:.2f},  dot(long, other) = {long_ @ other:.2f}")
print(f"  cosine both       = {angle_deg(short, other)[1]:.2f} / {angle_deg(long_, other)[1]:.2f}  <- identical")
print("  -> in semantic search always normalise, or a long document wins on length alone.")

a = [3. 4.], b = [4. 3.]
manual sum(a_i * b_i) = 24.0
np.dot(a, b)          = 24.0
a @ b                 = 24.0
(a * b).sum()         = 24.0   <- elementwise multiply THEN reduce

case                    dot      cos    angle
same direction       5.00     1.00     0.0°
orthogonal           0.00     0.00    90.0°
opposite            -5.00    -1.00   180.0°
45 degrees           1.00     0.71    45.0°

same direction, different magnitude:
  dot(short, other) = 2.00,  dot(long, other) = 200.00
  cosine both       = 1.00 / 1.00  <- identical
  -> in semantic search always normalise, or a long document wins on length alone.


## 3.2 Matrix Multiplication & Shape Rules

**RUN THE CELL BELOW** 👇

In [13]:
# Supply-chain analogy made concrete
#   A: factories x raw materials      B: raw materials x products
A = np.array([          # 2 factories, 3 raw materials
    [1., 2., 0.],
    [0., 1., 3.],
])
B = np.array([          # 3 raw materials, 4 products
    [1., 0., 2., 1.],
    [0., 1., 1., 0.],
    [2., 1., 0., 3.],
])

C = A @ B
print(f"A {A.shape} @ B {B.shape} = C {C.shape}   <- inner dims (3) cancel")
print(C)
print(f"\nC[0,2] by hand = {A[0,0]}*{B[0,2]} + {A[0,1]}*{B[1,2]} + {A[0,2]}*{B[2,2]} = {C[0,2]}\n")

# The shape error every beginner hits
try:
    B @ A          # (3,4) @ (2,3) -> 4 != 2
except ValueError as e:
    print(f"Shape error: {e}")
    print(f"  FIX: transpose -> B.T @ A.T works: {(B.T @ A.T).shape}")
    print("  NOTE: (A@B).T == B.T @ A.T  (order reverses)\n")

# A neural network layer IS a matmul + bias + activation
np.random.seed(0)
batch, in_dim, hidden = 4, 6, 3
x = np.random.randn(batch, in_dim)      # (4, 6)
W = np.random.randn(in_dim, hidden)     # (6, 3)
bias = np.random.randn(hidden)          # (3,) broadcast over the batch

h = np.maximum(x @ W + bias, 0)         # ReLU
print(f"one dense layer:  x{x.shape} @ W{W.shape} + b{bias.shape} -> h{h.shape}")
print(h.round(3))

# BATCHED matmul: (batch, m, n) @ (batch, n, p) -> (batch, m, p)
Ab = np.random.randn(8, 4, 5)
Bb = np.random.randn(8, 5, 6)
print(f"\nbatched: {Ab.shape} @ {Bb.shape} -> {np.matmul(Ab, Bb).shape}")
print("  the leading axes broadcast; the last two do the real matmul.")

# np.dot vs @ vs np.matmul on 1D/2D — know the difference
v = np.array([1., 2., 3.])
M = np.array([[1., 0., 0.], [0., 2., 0.], [0., 0., 3.]])
print(f"\nv @ M   = {v @ M}   (1D treated as a ROW vector)")
print(f"M @ v   = {M @ v}   (1D treated as a COLUMN vector)")
print("  @ and np.matmul agree on 2D+; np.dot differs for ndim>2 — prefer @.")

A (2, 3) @ B (3, 4) = C (2, 4)   <- inner dims (3) cancel
[[1. 2. 4. 1.]
 [6. 4. 1. 9.]]

C[0,2] by hand = 1.0*2.0 + 2.0*1.0 + 0.0*0.0 = 4.0

Shape error: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 4)
  FIX: transpose -> B.T @ A.T works: (4, 2)
  NOTE: (A@B).T == B.T @ A.T  (order reverses)

one dense layer:  x(4, 6) @ W(6, 3) + b(3,) -> h(4, 3)
[[0.529 2.859 0.305]
 [0.    0.    0.   ]
 [1.306 3.094 0.   ]
 [0.    0.811 1.337]]

batched: (8, 4, 5) @ (8, 5, 6) -> (8, 4, 6)
  the leading axes broadcast; the last two do the real matmul.

v @ M   = [1. 4. 9.]   (1D treated as a ROW vector)
M @ v   = [1. 4. 9.]   (1D treated as a COLUMN vector)
  @ and np.matmul agree on 2D+; np.dot differs for ndim>2 — prefer @.


## 3.3 Scaled Dot-Product Attention from Scratch

`Q @ K.T / sqrt(d_k)` → softmax → `@ V`. এটাই ChatGPT-র ভেতরের ইঞ্জিন। ৬ লাইনের NumPy।

**RUN THE CELL BELOW** 👇

In [14]:
np.random.seed(7)
seq_len, d_k = 5, 8      # 5 tokens, 8-dim keys/queries

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)   # (seq, seq) token-to-token similarity
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)     # causal mask: cannot see the future
    weights = softmax(scores, axis=-1)               # each row sums to 1
    return weights @ V, weights                       # (seq, d_k)

out, w = attention(Q, K, V)
print(f"Q{Q.shape} @ K.T{K.T.shape} -> scores {(Q @ K.T).shape}   <- every token scores every token")
print(f"attention weights (rows sum to 1):\n{w.round(3)}")
print(f"row sums: {w.sum(axis=1).round(6)}")
print(f"output shape: {out.shape}\n")

# WHY divide by sqrt(d_k)? Without it, dot products grow with dimension,
# softmax saturates, and gradients vanish. Watch the entropy collapse:
for scale, label in [(1.0, 'NO scaling  '), (np.sqrt(d_k), 'sqrt(d_k)   ')]:
    s = (Q @ K.T) / scale
    p = softmax(s, axis=-1)
    print(f"  {label}: max weight {p.max():.3f}, score std {s.std():.2f}")

# Causal mask — lower-triangular, what makes GPT autoregressive
causal = np.tril(np.ones((seq_len, seq_len), dtype=bool))
_, w_masked = attention(Q, K, V, mask=causal)
print(f"\ncausal attention weights (upper triangle is exactly 0):\n{w_masked.round(3)}")

# ANALOGY: attention = a meeting where every person (token) asks every other
#          'how relevant are you to me?' (Q@K.T), the answers are normalised
#          into a budget of 100% (softmax), then everyone's opinion (V) is
#          blended according to that budget.

Q(5, 8) @ K.T(8, 5) -> scores (5, 5)   <- every token scores every token
attention weights (rows sum to 1):
[[0.108 0.281 0.031 0.178 0.402]
 [0.064 0.15  0.059 0.214 0.513]
 [0.48  0.156 0.163 0.119 0.083]
 [0.372 0.104 0.095 0.394 0.034]
 [0.052 0.027 0.098 0.739 0.084]]
row sums: [1. 1. 1. 1. 1.]
output shape: (5, 8)

  NO scaling  : max weight 0.994, score std 2.53
  sqrt(d_k)   : max weight 0.739, score std 0.89

causal attention weights (upper triangle is exactly 0):
[[1.    0.    0.    0.    0.   ]
 [0.298 0.702 0.    0.    0.   ]
 [0.601 0.195 0.204 0.    0.   ]
 [0.386 0.108 0.099 0.408 0.   ]
 [0.052 0.027 0.098 0.739 0.084]]


---

# 4. Capstone — Build a Similarity Engine

তিনটি টপিক একসাথে: `ndarray` (embeddings ধরে রাখা) + vectorization (লুপ ছাড়া সব ডকুমেন্ট স্কোর করা) + dot product (cosine similarity)।

**RUN THE CELL BELOW** 👇

In [15]:
np.random.seed(42)

docs = [
    "neural networks learn patterns from data",
    "deep learning models need gpu memory",
    "gpu memory limits how large a model can be",
    "the cat sat on the mat",
    "dogs and cats are popular pets",
    "training a large model needs a lot of data",
]

# --- toy embedding: bag-of-words over the shared vocabulary ---
vocab = sorted({w for d in docs for w in d.split()})
vocab_index = {w: i for i, w in enumerate(vocab)}

def embed(text):
    v = np.zeros(len(vocab), dtype=np.float32)
    for w in text.split():
        if w in vocab_index:
            v[vocab_index[w]] += 1.0
    return v

E = np.stack([embed(d) for d in docs])          # (n_docs, vocab_size)
print(f"embedding matrix E: shape {E.shape}, dtype {E.dtype}, {E.nbytes} bytes\n")

# --- normalise ONCE, vectorized: every row becomes unit length ---
norms = np.linalg.norm(E, axis=1, keepdims=True)   # (n_docs, 1)
E_unit = E / np.maximum(norms, 1e-9)               # broadcasting, no loop

# --- the whole engine is ONE matmul ---
# unit vectors -> dot product IS cosine similarity
S = E_unit @ E_unit.T                              # (n_docs, n_docs)

print("pairwise cosine similarity matrix:")
print(S.round(2))
print(f"  diagonal is all 1.0 (a doc is identical to itself): {np.allclose(np.diag(S), 1.0)}\n")

def search(query, top_k=3):
    q = embed(query)
    q = q / max(np.linalg.norm(q), 1e-9)
    scores = E_unit @ q                            # (n_docs,) — one matmul, zero loops
    order = np.argsort(-scores)[:top_k]            # argsort gives INDICES, negate for descending
    return [(docs[i], float(scores[i])) for i in order]

for query in ["gpu memory model", "cats and dogs", "learn from data"]:
    print(f"query: {query!r}")
    for doc, score in search(query):
        print(f"   {score:.3f}  {doc}")
    print()

# --- proof that vectorization matters at scale ---
n_docs, dim = 20_000, 256
big_E = np.random.randn(n_docs, dim).astype(np.float32)
big_E /= np.linalg.norm(big_E, axis=1, keepdims=True)
q = big_E[0]

t0 = time.perf_counter()
loop_scores = np.array([float(np.dot(big_E[i], q)) for i in range(n_docs)])
t_loop = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
vec_scores = big_E @ q
t_vec = (time.perf_counter() - t0) * 1000

print(f"searching {n_docs:,} docs x {dim} dims")
print(f"  python loop of np.dot : {t_loop:7.2f} ms")
print(f"  single matmul         : {t_vec:7.2f} ms   ({t_loop / t_vec:.0f}x faster)")
print(f"  identical results     : {np.allclose(loop_scores, vec_scores, atol=1e-5)}")
print(f"  memory of E           : {big_E.nbytes / 1024**2:.1f} MB as float32 "
      f"(would be {big_E.nbytes * 2 / 1024**2:.1f} MB as float64)")

embedding matrix E: shape (6, 34), dtype float32, 816 bytes

pairwise cosine similarity matrix:
[[1.   0.   0.   0.   0.   0.12]
 [0.   1.   0.27 0.   0.   0.  ]
 [0.   0.27 1.   0.   0.   0.4 ]
 [0.   0.   0.   1.   0.   0.  ]
 [0.   0.   0.   0.   1.   0.  ]
 [0.12 0.   0.4  0.   0.   1.  ]]
  diagonal is all 1.0 (a doc is identical to itself): True

query: 'gpu memory model'
   0.577  gpu memory limits how large a model can be
   0.471  deep learning models need gpu memory
   0.174  training a large model needs a lot of data

query: 'cats and dogs'
   0.707  dogs and cats are popular pets
   0.000  neural networks learn patterns from data
   0.000  deep learning models need gpu memory

query: 'learn from data'
   0.707  neural networks learn patterns from data
   0.174  training a large model needs a lot of data
   0.000  deep learning models need gpu memory

searching 20,000 docs x 256 dims
  python loop of np.dot :    8.94 ms
  single matmul         :    0.45 ms   (20x faster)
  i

---

## Class 4 এ আমরা যা যা শিখলাম

| Achievement | Knowledge Applied |
| --- | --- |
| **Build a Similarity Engine** | NumPy arrays, vectorization, dot products, cosine similarity |

### Cheat Sheet

| Need | Code |
| --- | --- |
| Inspect an array | `a.shape`, `a.dtype`, `a.strides`, `a.nbytes` |
| Is it a view? | `np.shares_memory(a, b)`, `a.base` |
| Force a copy | `a.copy()` |
| Add an axis for broadcasting | `a[:, None]`, `keepdims=True` |
| Reduce along an axis | `a.sum(axis=0)` = per column, `axis=1` = per row |
| Matrix multiply | `A @ B`, batched `np.matmul(A, B)` |
| Cosine similarity | normalise rows, then `A @ B.T` |
| Stable softmax | subtract `max` before `np.exp` |

### Traps to Avoid

1. `for` loop over a batch — vectorize instead (~200x).
2. Mutating a slice and corrupting the source — slices are **views**.
3. `(4,3) + (4,)` shape error — broadcasting aligns from the **right**; use `[:, None]`.
4. Comparing raw dot products of different-length vectors — normalise first.
5. `float64` everywhere — `float32` halves memory and is enough for most models.

> **Valid Point:** PyTorch tensor = GPU-accelerated NumPy array + autograd. NumPy-তে মাস্টার হলে PyTorch ডালভাত।

---